# RAG：BM25、Dense、RRF 与分层评测

使用可控 embedding 观察排序，不下载模型。生产实验只需替换 EmbeddingModel，ACL、融合和指标保持相同。

In [ ]:
from collections.abc import Sequence

import numpy as np

from about_llm.rag import BM25Index, DenseIndex, Document, reciprocal_rank_fusion

documents = [
    Document('kv', 'KV Cache 保存历史 token 的 key 和 value', 'course'),
    Document('rag', 'RAG 先检索证据再生成回答', 'course'),
    Document('lora', 'LoRA 学习低秩权重增量', 'course'),
    Document('secret', '另一个租户的机密 RAG 文档', 'other'),
]

class ControlledEmbedder:
    def __init__(self, values): self.values = values
    def encode(self, texts: Sequence[str]):
        return np.asarray([self.values[text] for text in texts], dtype=np.float32)

vectors = {
    documents[0].text: (1.0, 0.0, 0.0),
    documents[1].text: (0.0, 1.0, 0.0),
    documents[2].text: (0.0, 0.0, 1.0),
    documents[3].text: (0.0, 1.0, 0.0),
    '如何避免重复计算注意力？': (0.9, 0.1, 0.0),
    '怎样用外部证据回答？': (0.1, 0.9, 0.0),
}
bm25 = BM25Index(documents)
dense = DenseIndex(documents, ControlledEmbedder(vectors))


In [ ]:
query = '怎样用外部证据回答？'
lexical = bm25.search(query, tenant_id='course', top_k=3)
semantic = dense.search(query, tenant_id='course', top_k=3)
fused = reciprocal_rank_fusion([lexical, semantic], top_k=3)

def show(name, results):
    print(name, [(item.document.document_id, round(item.score, 4)) for item in results])

show('BM25', lexical)
show('Dense', semantic)
show('RRF', fused)
assert all(item.document.tenant_id == 'course' for item in fused)


In [ ]:
from about_llm.evaluation import mean_reciprocal_rank, recall_at_k

queries = {
    'q-kv': '如何避免重复计算注意力？',
    'q-rag': '怎样用外部证据回答？',
}
relevant = {'q-kv': {'kv'}, 'q-rag': {'rag'}}
retrieved = {}
for query_id, text in queries.items():
    dense_results = dense.search(text, tenant_id='course', top_k=2)
    retrieved[query_id] = [item.document.document_id for item in dense_results]

print('retrieved:', retrieved)
print('Recall@2:', recall_at_k(retrieved, relevant, k=2))
print('MRR@2:', mean_reciprocal_rank(retrieved, relevant, k=2))
assert recall_at_k(retrieved, relevant, k=2) == 1.0
assert mean_reciprocal_rank(retrieved, relevant, k=2) == 1.0


## 继续实验

把 ControlledEmbedder 换成固定 revision 的 sentence-transformers 模型；扩大 query 与相关性标注；分别比较 BM25、dense、RRF 和 reranker。最终答案错误时仍要区分召回、重排、上下文和生成。